In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from utils import plot_returns, print_metrics

## Asset Universe

The portfolio is built from four diversified asset classes, each represented by a liquid instrument:

| Ticker | Asset Class | Description |
|---|---|---|
| `ES=F` | Equities | S&P 500 E-mini Futures — tracks US large-cap equity performance |
| `ZN=F` | Fixed Income | 10-Year US Treasury Note Futures — proxy for long-duration government bonds |
| `GC=F` | Commodities | Gold Futures — inflation hedge and safe-haven asset |
| `DX-Y.NYB` | Currencies | ICE US Dollar Index — measures the dollar against a basket of major currencies |

These four assets have historically low or negative correlations with each other, making them well-suited for a **risk parity** strategy.

In [ ]:
# Download front-month futures data of S&P500, 10-year Treasuries, gold and US dollar
symbols = ["ES=F", "ZN=F", "GC=F", "DX-Y.NYB"]
data = yf.download(symbols, period="10y")
# Resample data so that we deal with monthly data instead of daily to reduce noise
data = data.resample("ME").last()
data.index = pd.to_datetime(data.index)
# Subset adjusted close prices and fill NaNs with value know at time t
# Drop rows with unknown prices in the beginning of the dataset
prices = data["Close"].ffill().dropna()
prices.index = pd.to_datetime(prices.index)

In [ ]:
prices

In [ ]:
# Compute logarithmic returns
log_returns = np.log(prices).diff()

In [ ]:
log_returns.head()

In [ ]:
def compute_risk_parity_weights(returns, window_size=36):
    # compute volatility known at time t
    # std of log returns directly gives volatility — no exponentiation needed
    rolling_vol = returns.rolling(window_size).std()
    rolling_inverse_vol = 1 / rolling_vol
    # divide inverse volatility by the sum of inverse volatilities
    risk_parity_weights = rolling_inverse_vol.apply(lambda row: row / row.sum(), axis=1)
    return risk_parity_weights

In [ ]:
# Compute risk parity weights
risk_parity_weights = compute_risk_parity_weights(log_returns)
# shift weights by one period to use only information available at time t
risk_parity_weights = risk_parity_weights.shift(1)

# Approximation considering small monthly returns
weighted_returns_approximated = (log_returns * risk_parity_weights).sum(axis=1)

# Exact calculation of the returns
simple_returns = np.exp(log_returns) - 1
weighted_returns_exact = (simple_returns * risk_parity_weights).sum(axis=1)

In [ ]:
print_metrics(weighted_returns_exact)
plot_returns(weighted_returns_exact)

In [ ]:
print_metrics(weighted_returns_approximated)
plot_returns(weighted_returns_approximated)